# agentic-verifier-selfrefine — follow-up runs for the paper

Three pieces of GPU work the paper still needs, in order of value. Run top to bottom;
every stage backs up to Drive (`MyDrive/agentic_backups/`) **without model weights**.

| section | what | why | time |
|---|---|---|---|
| A | restore the selected checkpoint from Drive, re-score it | identity check: must reproduce SFT 0.333 / SFT+DPO 0.444 | 6 min |
| B | agent loop on the selected checkpoint | section 17 of the clean run left no output | 20 min |
| C | extract the selected-checkpoint ladder CSV from the 1.8 GB backup | the paper can only quote point estimates for it today | 1 min |
| D | four SFT→DPO passes on the **balanced v2** preference data | tests the remedy for the `no solution` displacement (bug 8) | 50 min |

Nothing here overwrites a recorded result: each stage writes its own directory under
`results/raw/`.

## 1. GPU, upload, unpack, install

In [ ]:
!nvidia-smi
import glob, os
from google.colab import files
for z in glob.glob('/content/agentic-verifier-selfrefine*.zip'):
    os.remove(z)
uploaded = files.upload()          # pick agentic-verifier-selfrefine.zip
print('uploaded:', list(uploaded))

In [ ]:
import glob, os, shutil
ZIP_PATH = sorted(glob.glob('/content/agentic-verifier-selfrefine*.zip'), key=os.path.getmtime)[-1]
shutil.rmtree('/content/agentic-verifier-selfrefine', ignore_errors=True)
!unzip -q -o "{ZIP_PATH}" -d /content
os.chdir('/content/agentic-verifier-selfrefine')
assert os.path.exists('data/dpo_pairs_zeroshot_v2.jsonl'), 'this zip predates the v2 data -- use the new one'
!pip -q uninstall -y torchao
!pip -q install -r requirements-colab.txt -r requirements-dev.txt
assert globals().get('_exit_code', 0) == 0, 'pip install failed'
!python -m pytest -q
assert globals().get('_exit_code', 0) == 0, 'test suite failed -- stop here' 

## 2. Helpers and Drive

In [ ]:
import os, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/agentic-verifier-selfrefine'
DRIVE_DIR = '/content/drive/MyDrive/agentic_backups'
os.makedirs(DRIVE_DIR, exist_ok=True)

def check(stage):
    code = globals().get('_exit_code', 0)
    assert code == 0, f'{stage} FAILED (exit {code}). Stop here and send the output above.'
    print(f'OK: {stage}')

def backup(tag):
    os.makedirs('/content/out', exist_ok=True)
    path = f'/content/out/followup_{tag}.zip'
    if os.path.exists(path):
        os.remove(path)
    subprocess.run(['zip', '-rq', path, 'results', '-x', 'results/sft-merged-model/*',
                    'results/lora-*/*', 'results/selected-checkpoint/*'], cwd=ROOT, check=True)
    size_mb = os.path.getsize(path) / 1e6
    assert size_mb < 50, f'{path} is {size_mb:.0f} MB -- model weights leaked into the backup'
    shutil.copy(path, DRIVE_DIR)
    print(f'backup: {path} ({size_mb:.1f} MB) -> Drive')

## A. Restore the selected checkpoint and re-score it

The clean run copied `results/` to `MyDrive/agentic_backups/results_snapshot/`, which
holds `selected-checkpoint/`. Greedy decoding of the same weights on the same GPU type
should reproduce the recorded 0.333 / 0.444. If it does not, stop: B would not be
measuring the model the paper describes.

In [ ]:
src = None
for cand in (os.path.join(DRIVE_DIR, 'selected-checkpoint'),
             os.path.join(DRIVE_DIR, 'results_snapshot', 'selected-checkpoint')):
    if os.path.exists(os.path.join(cand, 'sft-merged-model')):
        src = cand
        break
assert src, 'selected-checkpoint not found on Drive'
for sub in ('sft-merged-model', 'lora-dpo-zeroshot-adapter'):
    dst = os.path.join(ROOT, 'results', sub)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(os.path.join(src, sub), dst)
print('restored from', src)

In [ ]:
!cd /content/agentic-verifier-selfrefine/src && AVS_EVAL_DIR=../results/raw/selected_eval_restored python evaluate_redesign_v2.py
check('re-score of the restored checkpoint')
import glob, json
s = json.load(open(sorted(glob.glob(os.path.join(ROOT, 'results/raw/selected_eval_restored/summary_*.json')))[-1]))
print('SFT', round(s['sft'], 3), '| SFT+DPO', round(s['sft_dpo'], 3))
assert abs(s['sft'] - 0.333) < 0.03 and abs(s['sft_dpo'] - 0.444) < 0.03, (
    'restored checkpoint does not reproduce the recorded score -- stop and send this output')

## B. Agent loop on the selected checkpoint

Verifier feedback vs a re-sampling control, 45 problems × 3 seeds, paired seeds.

In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python run_llm_agent_experiment.py
check('agent loop on the selected checkpoint')
dst = os.path.join(ROOT, 'results/raw/selected_agent')
os.makedirs(dst, exist_ok=True)
for f in glob.glob(os.path.join(ROOT, 'results/llm_agent_experiment*')):
    shutil.move(f, dst)
print(sorted(os.listdir(dst)))
backup('B_selected_agent')

## C. The selected-checkpoint ladder CSV, from the big backup

Reads one member out of `results_7_after_selected_ladder.zip` without unpacking the
weights.

In [ ]:
import zipfile
big = os.path.join(DRIVE_DIR, 'results_7_after_selected_ladder.zip')
dst = os.path.join(ROOT, 'results/raw/selected_ladder')
os.makedirs(dst, exist_ok=True)
with zipfile.ZipFile(big) as z:
    names = [n for n in z.namelist() if os.path.basename(n).startswith('feedback_protocol_ablation')
             and '/raw/' not in n and '/recorded/' not in n]
    print(names)
    for n in names:
        with z.open(n) as fsrc, open(os.path.join(dst, os.path.basename(n)), 'wb') as fdst:
            shutil.copyfileobj(fsrc, fdst)
print(sorted(os.listdir(dst)))
backup('C_selected_ladder')

## D. DPO on the balanced v2 preference data, four passes

v1 (every recorded run) has no `no solution` pair on either side, and DPO on it answers
`no solution` to most solvable quadratics. v2 adds the 25 no-real-root problems and an
equal number of solvable problems whose rejected answer is `no solution`. Same SFT
recipe, same DPO config, only the data path changes. If the displacement is caused by
the coverage gap, `no solution` on solvable problems should fall back toward SFT's 5%.

In [ ]:
!cd /content/agentic-verifier-selfrefine/src && AVS_DATA_PATH=../data/dpo_pairs_zeroshot_v2.jsonl AVS_EVAL_DIR=../results/raw/eval_v2 python rerun_all.py sft dpo eval --repeat=4
check('four v2 passes')
!cd /content/agentic-verifier-selfrefine/src && python summarize_eval.py --dir ../results/raw/eval_v2
backup('D_dpo_v2')

## Done

Everything is in `MyDrive/agentic_backups/followup_*.zip` (each < 1 MB). Send those and
this notebook with outputs; `src/paper_analysis.py` picks the new directories up once
they are copied into `results/recorded/`.